In [ ]:
from __future__ import annotations

import os
import re
import json
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple, TypedDict
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

import pandas as pd
import pdfplumber
from pydantic import BaseModel, Field, ValidationError

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import Markdown, display
from langchain_teddynote import logging

# 추적을 위한 프로젝트 이름 설정
logging.langsmith("Samsung-Asset-AI-Portal")

# prompt 모듈에서 필요한 항목 import
from prompt import (
    _SYSTEM_PROMPT,
    _SYSTEM_PROMPT_ENG,
    get_prompt_pdf_text_to_markdown,
    get_prompt_text_to_markdown,
    get_prompt_pdf_text_to_markdown_validate,
    get_prompt_text_to_markdown_validate,
    get_prompt_confirmed_expected_clarification,
    get_prompt_text_to_markdown_validate_report,
    get_prompt_text_to_markdown_error_fix,
    get_prompt_confirmed_expected_clarification_validate_report,
    get_prompt_confirmed_expected_clarification_error_fix,
    get_prompt_text_to_markdown_with_validate,
    get_prompt_pdf_to_markdown_with_validate
)

# document_parser 모듈에서 필요한 함수들 import
from document_parser import (
    extract_pdf_with_docling,
    extract_pdf_with_pdfplumber,
    parser_excel,
    get_file_path,
    get_last_ai_message
)

from utils import (
    format_messages,
    extract_validate_result,
    has_validate_result,
    check_validate_result,
)

from llm_util import StreamPrinter


load_dotenv(override=True)


### Util

In [ ]:
def visualize_res_result(res: Dict[str, Any]):
    """
    extract_one_file의 결과값 res를 테이블 형태로 시각화하는 함수
    
    Args:
        res: extract_one_file 함수의 반환값 (file, db_rows, ingest_issues, extract_issues, validation_report 포함)
    """
    if not res:
        print("res 변수가 비어있습니다.")
        return
    
    # 파일 정보 표시
    print("=" * 80)
    print(f"📄 파일: {res.get('file', 'N/A')}")
    print("=" * 80)
    print()
    
    # db_rows 테이블 생성
    if res.get('db_rows'):
        db_rows_data = []
        for idx, row in enumerate(res['db_rows'], 1):
            row_data = {'Row #': idx}
            # 각 행의 모든 키-값을 추가
            for key, value in row.items():
                # date와 Decimal 타입을 문자열로 변환
                if isinstance(value, date):
                    row_data[key] = value.isoformat()
                elif isinstance(value, Decimal):
                    row_data[key] = str(value)
                else:
                    row_data[key] = value
            db_rows_data.append(row_data)
        
        if db_rows_data:
            df_db_rows = pd.DataFrame(db_rows_data)
            print("=" * 80)
            print("📊 DB 행 데이터")
            print("=" * 80)
            # 모든 행과 열을 표시하도록 pandas 옵션 설정
            with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None, 'display.max_colwidth', None):
                display(df_db_rows)
            print()
    else:
        print("=" * 80)
        print("📊 DB 행 데이터: 없음")
        print("=" * 80)
        print()
    
    # ingest_issues 표시
    if res.get('ingest_issues'):
        print("=" * 80)
        print("⚠️  Ingest 이슈")
        print("=" * 80)
        for idx, issue in enumerate(res['ingest_issues'], 1):
            print(f"{idx}. {issue}")
        print()
    else:
        print("=" * 80)
        print("✅ Ingest 이슈 없음")
        print("=" * 80)
        print()
    
    # extract_issues 표시
    if res.get('extract_issues'):
        print("=" * 80)
        print("⚠️  Extract 이슈")
        print("=" * 80)
        for idx, issue in enumerate(res['extract_issues'], 1):
            print(f"{idx}. {issue}")
        print()
    else:
        print("=" * 80)
        print("✅ Extract 이슈 없음")
        print("=" * 80)
        print()
    
    # validation_report 표시
    validation_report = res.get('validation_report', '')
    print("=" * 80)
    print("🔍 검증 리포트")
    print("=" * 80)
    if validation_report:
        print(validation_report)
    else:
        print("검증 리포트 없음")
    print()

In [ ]:
# ============================================================
# 1) LLM 설정
# ============================================================
LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

_SYSTEM_PROMPT = (
    "너는 자산운용사 변액일임펀드 설정/해지 지시서에서 "
    "DB 적재용 구조화 데이터를 추출하는 전문가다. "
    "추측하지 말고, 문서 근거가 없는 값은 만들지 마라."
)

### LLM 생성

In [ ]:
def create_llm_model():

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

### DB 적재용 스키마 클래스 

In [ ]:
# ============================================================
# 2) 목표 테이블(tb_variable_order_data) 적재용 Row 모델
# ============================================================
class DBRow(BaseModel):
    task_id: int
    fund_code: str
    fund_name: str
    settle_class: Literal["CONFIRMED", "PENDING"]
    order_type: Literal["SUB", "RED"]
    base_date: date
    t_day: Optional[int]
    transfer_amount: Decimal  # DECIMAL(18,2)


# ============================================================
# 3) LLM이 출력하는 "값만(Value-only) 추출" 스키마
# ============================================================
class ExtractedOrder(BaseModel):
    fund_code: str
    fund_name: str
    settle_class: Literal["CONFIRMED", "PENDING"]
    order_type: Literal["SUB", "RED"]
    base_date: Optional[str] = Field(description="YYYY-MM-DD or null")
    t_day: Optional[int] = Field(description="CONFIRMED=0, PENDING=1..N or null")
    transfer_amount: str = Field(description="금액 문자열 (콤마/부호 포함 가능)")

class ExtractionValueOnly(BaseModel):
    orders: List[ExtractedOrder]
    issues: List[str] = Field(default_factory=list)

### 파싱 유틸 (금액, 날짜, 파일명에서 base_date 추출)

In [ ]:
# ============================================================
# 4) 파싱 유틸 (금액/날짜/파일명 base_date 추정)
# ============================================================
def parse_decimal_amount(s: str) -> Decimal:
    """
    문자열 금액 -> Decimal(18,2)
    - 콤마 제거
    - 괄호 음수 지원
    - 부호 유지
    """
    s = (s or "").strip()
    if not s:
        return Decimal("0.00")

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1].strip()

    s = re.sub(r"[^\d\-,\.]", "", s)
    if s.startswith("-"):
        neg = True
        s = s[1:]

    s = s.replace(",", "")
    if s == "" or s == ".":
        return Decimal("0.00")

    val = Decimal(s) if "." in s else Decimal(s)
    if neg:
        val = -val
    return val.quantize(Decimal("0.01"))

def abs_decimal(d: Decimal) -> Decimal:
    return (d if d >= 0 else -d).quantize(Decimal("0.01"))

def infer_order_type_from_amount(amount: Decimal) -> Literal["SUB", "RED"]:
    return "SUB" if amount >= 0 else "RED"

def guess_base_date_from_filename(path: str) -> Optional[date]:
    """
    파일명에 yymmdd(예: 250826)가 있으면 2025-08-26으로 추정.
    문서에 base_date가 없을 때만 보조적으로 사용.
    """
    name = Path(path).name
    m = re.search(r"(\d{6})", name)
    if not m:
        return None
    yymmdd = m.group(1)
    yy, mm, dd = int(yymmdd[0:2]), int(yymmdd[2:4]), int(yymmdd[4:6])
    year = 2000 + yy  # 업무 관례상 20xx 가정
    try:
        return date(year, mm, dd)
    except ValueError:
        return None

### 문서 파일 파서, markdown 변환(코드 레벨) - pdf, 엑셀

In [ ]:
# ============================================================
# 5) Ingest: PDF/XLS/XLSX -> 정규화 Markdown 생성
# ============================================================
def df_to_markdown(df: pd.DataFrame, max_rows: int = 2000) -> str:
    if df is None:
        return ""
    df = df.fillna("")
    if len(df) > max_rows:
        df = df.head(max_rows)
    return df.to_markdown(index=False)

def extract_excel_to_markdown(xls_path: str, max_rows_per_sheet: int = 2000) -> Tuple[str, List[str]]:
    issues: List[str] = []
    chunks: List[str] = []
    try:
        xls = pd.ExcelFile(xls_path)
    except Exception as e:
        return "", [f"EXCEL_OPEN_FAIL:{type(e).__name__}:{e}"]

    for sheet in xls.sheet_names:
        try:
            # dtype=str 제거: 숫자/날짜 형식 깨짐 완화
            df = pd.read_excel(xls_path, sheet_name=sheet)
            md = df_to_markdown(df, max_rows=max_rows_per_sheet)
            chunks.append(f"\n\n# SHEET {sheet}\n{md}\n")
        except Exception as e:
            issues.append(f"SHEET_READ_FAIL:{sheet}:{type(e).__name__}:{e}")
            continue

    return "\n".join(chunks), issues

def extract_pdf_to_markdown(pdf_path: str, max_pages: int = 50) -> Tuple[str, List[str]]:
    issues: List[str] = []
    chunks: List[str] = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages[:max_pages]):
                text = page.extract_text() or ""
                chunks.append(f"\n\n# PAGE {i+1}\n{text}\n")

                # 가능한 경우 표도 추출해 Markdown으로 추가(표 구조 개선)
                try:
                    tables = page.extract_tables() or []
                    for ti, tbl in enumerate(tables):
                        # tbl: List[List[str]] 형태
                        if not tbl or len(tbl) < 2:
                            continue
                        df = pd.DataFrame(tbl[1:], columns=tbl[0])
                        md = df_to_markdown(df, max_rows=2000)
                        chunks.append(f"\n\n## PAGE {i+1} TABLE {ti+1}\n{md}\n")
                except Exception:
                    # 테이블 추출 실패는 무시(텍스트는 이미 확보)
                    pass
    except Exception as e:
        issues.append(f"PDF_OPEN_FAIL:{type(e).__name__}:{e}")
        return "", issues

    return "\n".join(chunks), issues



### excel 파서, markdown 변환(LLM 레벨) - pdf, 엑셀

In [ ]:
# LLM 모델을 사용하여 text 정보를 markdown으로 변환
# 싱글톤 

def text_to_markdown_with_llm(document_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_with_validate(document_text)
    print("########## text to markdown prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response



def convert_excel_to_markdown(file_path: str) -> Tuple[str, List[str]]:
    issues: List[str] = []
    # 엑셀 파일 경로 확인
    if not os.path.exists(file_path):
        # raise FileNotFoundError(f"File not found: {file_path}")
        issues.append(f"File not found: {file_path}")
        return "", issues
    try:
        # 엑셀 파일에서 텍스트 추출
        # _original_text = parser_excel(file_path)
        _original_text, issues = extract_excel_to_markdown(file_path)

        # 텍스트를 markdown으로 변환
        # response_markdown = text_to_markdown_with_plan(document_text_excel)
        response_md = text_to_markdown_with_llm(_original_text)
        response_markdown = response_md.content
        print("########## convert_excel_to_markdown ##########")
        print(response_markdown)
        # markdown_text = get_last_ai_message(response_markdown)
    except Exception as e:
        issues.append(f"PDF_OPEN_FAIL:{type(e).__name__}:{e}")
        return "", issues

    return response_markdown, issues


### pdf 파서, markdown 변환(LLM 레벨) - pdf, 엑셀

In [ ]:
# LLM을 사용하여 pdf를 markdown으로 변환
from sys import excepthook


def pdf_to_markdown_with_llm(docling_text: str, pdfplumber_text):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_pdf_to_markdown_with_validate(docling_text, pdfplumber_text)
    print("########## pdf to markdown prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


def convert_pdf_to_markdown(file_path: str, password: str) -> Tuple[str, List[str]]:
    issues: List[str] = []
    # 엑셀 파일 경로 확인
    if not os.path.exists(file_path):
        # raise FileNotFoundError(f"File not found: {file_path}")
        issues.append(f"File not found: {file_path}")
        return "", issues
    try:
        # 엑셀 파일에서 텍스트 추출
        # _original_text = parser_excel(file_path)
        docling_text = ""
        try:
            docling_text = extract_pdf_with_docling(file_path, password)
        except Exception as e:
            pass
        pdfplumber_text = extract_pdf_with_pdfplumber(file_path, password)

        # 텍스트를 markdown으로 변환
        # response_markdown = text_to_markdown_with_plan(document_text_excel)
        response_md = pdf_to_markdown_with_llm(docling_text, pdfplumber_text)
        response_markdown = response_md.content
        print("########## convert_pdf_to_markdown ##########")
        print(response_markdown)
        # markdown_text = get_last_ai_message(response_markdown)
    except Exception as e:
        issues.append(f"PDF_OPEN_FAIL:{type(e).__name__}:{e}")
        return "", issues

    return response_markdown, issues

### 문서 마크다운 변환 - 확장자별 분기

In [ ]:
def ingest_file_to_markdown(path: str, password: str) -> Tuple[str, List[str]]:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        # return extract_pdf_to_markdown(path)
        return convert_pdf_to_markdown(path, password)
    if ext in [".xlsx", ".xls"]:
        # return extract_excel_to_markdown(path)
        return convert_excel_to_markdown(path)
    return "", [f"UNSUPPORTED_EXT:{ext}"]

### Graph 상태 저장 클래스

In [ ]:
class RState(TypedDict):
    task_id: int
    file_path: str
    password: str

    markdown_doc: str
    ingest_issues: List[str]
    extract_prompt: str

    extracted: Optional[ExtractionValueOnly]
    validation_ok: bool
    validation_report: str
    iterations: int

    db_rows: List[Dict[str, Any]]
    extract_issues: List[str]

### Graph 생성 - 마크다운 변환

In [ ]:
# ============================================================
# 6) LangGraph Nodes (파일 1개 단위 실행)
# ============================================================
def node_ingest(state: RState) -> RState:
    md, issues = ingest_file_to_markdown(state["file_path"], state["password"])
    if md.strip():
        md = f"\n\n========== FILE: {Path(state['file_path']).name} ==========\n{md}\n"
    
    display(Markdown(md))

    return {**state, "markdown_doc": md, "ingest_issues": issues}

### Graph 생성 - 데이터 추출

In [ ]:
def build_extract_rule_prompt() -> str:
    return """
[출력 규칙 - 최우선]
- 출력은 반드시 JSON만 출력합니다. (설명/마크다운/코드펜스/여분 문자 금지)
- 아래 스키마를 반드시 만족해야 합니다:
  {{
    "orders":[
      {{
        "fund_code":"...",
        "fund_name":"...",
        "settle_class":"CONFIRMED|PENDING",
        "order_type":"SUB|RED",
        "base_date":"YYYY-MM-DD 또는 null",
        "t_day":0 또는 1..N 또는 null,
        "transfer_amount":"숫자 문자열(콤마/부호 포함 가능)"
      }}
    ],
    "issues":[...]
  }}

[추출 목표]
orders[]에는 DB row 후보만 넣습니다.
- 1 order = 1 펀드의 1건 이체(투입/인출)입니다.
- 같은 펀드에 대해 T일/예정일자(T+N)별 금액이 있으면 그만큼 order를 여러 개 만드세요.
- 각 펀드 행에서 T/T+N 컬럼의 투입/인출 금액이 0이 아니면 반드시 각각 row를 생성하세요.
- 금액이 작아도 0이상이면 생략 금지
- 표에 Subscription과 Redemption이 컬럼으로 구분되어 있으면 모두 처리하세요. 한쪽만 처리하면 안 됩니다.

[매핑 규칙]
- settle_class:
  - 당일/확정/실행/당일이체/당일투입/당일인출 -> CONFIRMED
  - 예정/청구/예상/T+N -> PENDING
- order_type:
  - 투입/설정/입금/매입 -> SUB
  - 인출/해지/출금/환매 -> RED
  - 금액 부호가 명시된 문서는 음수=RED, 양수=SUB를 우선 적용
  - 표에 '구분' 또는 'Transaction Type'이 있고 값이 'Subscription'이면 order_type=SUB
  - 표에 '구분' 또는 'Transaction Type'이 있고 값이 'Redemption'이면 order_type=RED
- base_date:
  - 기준일/기준일자/T일/결제일/settlement date/문서작성일 등으로 명시된 날짜를 YYYY-MM-DD로 추출
  - 문서에서 찾을 수 없으면 null (이 경우 issues에 BASE_DATE_MISSING을 기록)
- t_day:
  - CONFIRMED는 0
  - PENDING은 예정 컬럼 순서대로 1..N 방식으로 순번 부여 (날짜 차이 계산 금지), 아니면 null
- transfer_amount:
  - 문서의 금액을 숫자 문자열로 반환(콤마/부호 포함 가능)

[제외 규칙]
- "합산", "합계", "총계", "TOTAL", "summary", "순유입" 등 요약/합계 행은 절대 orders에 만들지 말 것.
- 요약/합계 컬럼 차단: 컬럼명에 합산, 합계, 총계, total, summary, 순유입 의미 포함 -> 금액 적재 금지
- 좌수 컬럼 차단: 컬럼명에 좌수 포함 → 금액 적재 금지
- NET(당일이체금액) 차단 조건: 입금액·출금액 둘 다 존재 & 당일이체금액 = (입금-출금) 형태면 → 당일이체금액 row 생성 금지
    """

def build_extract_prompt(doc_text: str, rule_prompt: str) -> str:
    # ✅ null 허용은 base_date에만(스키마가 Optional)
    # ✅ 출력 JSON only 강제
    return f"""
당신은 변액일임펀드 설정/해지 지시서에서 tb_variable_order_data 적재용 "주문(이체) 단위" 데이터를 추출합니다.

{rule_prompt}

[문서(정규화 markdown)]
{doc_text}
"""

def node_extract_value_only(state: RState) -> RState:
    doc_text = state["markdown_doc"]
    if not doc_text.strip():
        extracted = ExtractionValueOnly(orders=[], issues=["NO_TEXT"])
        print("==============node_extract_value_only error===============")
        print("no markdown text")
        return {**state, "extracted": extracted, "extract_issues": extracted.issues}

    llm = create_llm_model()
    extractor = llm.with_structured_output(ExtractionValueOnly)

    rule_prompt = build_extract_rule_prompt()
    prompt = build_extract_prompt(doc_text, rule_prompt)
    print("==============node_extract_value_only prompt===============")
    print(prompt)
    try:
        extracted: ExtractionValueOnly = extractor.invoke(
            [SystemMessage(_SYSTEM_PROMPT), HumanMessage(prompt)]
        )
        print("==============node_extract_value_only extracted===============")
        print(extracted)
    except Exception as e:
        extracted = ExtractionValueOnly(orders=[], issues=[f"LLM_EXTRACT_FAIL:{type(e).__name__}:{e}"])
        print("==============node_extract_value_only error===============")
        print("extract fail")
        return {**state, "extracted": extracted, "extract_issues": extracted.issues}

    # base_date가 null인 orders가 있으면, 파일명 추정(파일 단위 실행이므로 혼선 없음)
    issues = list(extracted.issues)
    guess = guess_base_date_from_filename(state["file_path"])
    print("==============guess_base_date_from_filename gusee===============")
    print(guess)
    for o in extracted.orders:
        if not o.base_date:
            if guess:
                o.base_date = guess.isoformat()
                issues.append("BASE_DATE_FROM_FILENAME")
            else:
                issues.append("BASE_DATE_MISSING")
    extracted.issues = issues

    return {**state, "extracted": extracted, "extract_issues": extracted.issues, "extract_prompt": prompt}
    


### Graph 생성 - 데이터 검증

In [ ]:
def node_validate_(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return {**state, "validation_ok": False, "validation_report": "NO_EXTRACTED"}

    problems: List[str] = []

    for i, o in enumerate(extracted.orders):
        if not o.fund_code or not o.fund_name:
            problems.append(f"ORDER[{i}]:MISSING_FUND")

        if o.settle_class not in ["CONFIRMED", "PENDING"]:
            problems.append(f"ORDER[{i}]:BAD_SETTLE_CLASS:{o.settle_class}")

        if o.order_type not in ["SUB", "RED"]:
            problems.append(f"ORDER[{i}]:BAD_ORDER_TYPE:{o.order_type}")

        # base_date format
        if not o.base_date:
            problems.append(f"ORDER[{i}]:MISSING_BASE_DATE")
        else:
            try:
                datetime.strptime(o.base_date, "%Y-%m-%d")
            except Exception:
                problems.append(f"ORDER[{i}]:BAD_BASE_DATE:{o.base_date}")

        # amount
        try:
            _ = parse_decimal_amount(o.transfer_amount)
        except Exception:
            problems.append(f"ORDER[{i}]:BAD_AMOUNT:{o.transfer_amount}")

        # t_day consistency
        if o.settle_class == "CONFIRMED":
            if o.t_day is not None and o.t_day != 0:
                problems.append(f"ORDER[{i}]:CONFIRMED_TDAY_NOT_ZERO:{o.t_day}")
        if o.settle_class == "PENDING":
            if o.t_day is not None and o.t_day < 1:
                problems.append(f"ORDER[{i}]:PENDING_TDAY_LT1:{o.t_day}")

    print("================node_validate==================")
    print(problems)
    if problems:
        return {**state, "validation_ok": False, "validation_report": "\n".join(problems)}

    return {**state, "validation_ok": True, "validation_report": "OK"}

def build_validate_prompt(markdown_doc: str, extracted):
    return f"""
    당신은 엄격한 검증자다. 

    아래는 변액일임펀드 설정/해지 지시서 원문과 원문에서 추출한 추출 결과이다.
    변액일임펀드 설정/해지 지시서 원문과 추출 결과를 비교 분석하여 추출 결과가 원문과 일치하고 DB 적재에 적합한지 검사하라.



    ## [검증 항목]
    1) 펀드 행 누락 가능성: 표/리스트에 있는 행을 빠뜨리지 않았는가?
    2) base_date가 YYYY-MM-DD 형식이며 원문에 존재하는가?
    3) funds에 합계/총계/설정합계/해지합계 같은 요약행이 섞이지 않았는가?
    4) 각 fund의 fund_code/fund_name이 비어있지 않은가?
    5) 확정분/청구분 분류가 정확한가?
    6) 설정/해지 분류가 정확한가?
    7) 날짜/금액/좌수 등 형식이 원문과 크게 어긋나지 않는가?
    8) t_day가 정확한가?



    ## [검사 보고 지침]
    - 검사 보고는 수정 조치가 필요한 문제점만 작성할 것
    - 수정 조치가 필요없는 항목은 절대 작성하지 말 것!



    ## [출력 형식(반드시 지킬 것)]
    - validation_ok: true 또는 false
    - report: 문제점 bullet list 또는 OK



    ## [변액일임펀드 설정/해지 지시서 원문]
    {markdown_doc}



    ## [현재 추출 결과(JSON)]
    {extracted.model_dump_json()}
    """ 

def node_validate(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return {**state, "validation_ok": False, "validation_report": "NO_EXTRACTED"}

    markdown_doc = state.get("markdown_doc")
    human_prompt = build_validate_prompt(markdown_doc, extracted)
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    print("########## node_validate prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환
    report = response.content.strip()

    print("########## node_validate report ##########")
    print(report)

    ok = False
    low = report.lower()
    if "validation_ok: true" in low or report == "OK":
        ok = True

    return {**state, "validation_ok": ok, "validation_report": report}

### Graph 생성 - 데이터 검증 오류 수정

In [ ]:
def node_repair(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return state

    llm = create_llm_model()
    extractor = llm.with_structured_output(ExtractionValueOnly)
    rule_prompt = build_extract_rule_prompt()

    prompt = f"""
너는 변액일임펀드 지시서 추출 결과를 "오류 항목만" 수정하는 보정 에이전트다.
아래 validation_report의 문제만 해결하여 orders를 수정하라. 다른 항목은 임의 변경 금지.

[validation_report]
{state["validation_report"]}

[현재 추출 결과(JSON)]
{extracted.model_dump_json()}

[문서(정규화 markdown)]
{state["markdown_doc"]}

[출력 규칙]
- JSON만 출력 (설명/마크다운/코드펜스 금지)
- orders 구조 유지, 문제 항목만 수정
- 합계/총계/요약행을 DB row로 만들지 말 것
- 합계/총계/요약/순유입 컬럼은 DB row에 포함하지 말 것
"""
    print("=============node_repair prompt=============")
    print(prompt)
    fixed: ExtractionValueOnly = extractor.invoke([SystemMessage(_SYSTEM_PROMPT), HumanMessage(prompt)])

    # 보정 후에도 base_date가 비면 파일명 추정 보강
    issues = list(fixed.issues)
    guess = guess_base_date_from_filename(state["file_path"])
    for o in fixed.orders:
        if not o.base_date:
            if guess:
                o.base_date = guess.isoformat()
                issues.append("BASE_DATE_FROM_FILENAME")
            else:
                issues.append("BASE_DATE_MISSING")
    fixed.issues = issues

    return {**state, "extracted": fixed, "iterations": state["iterations"] + 1, "extract_issues": fixed.issues}

### Graph 생성 - 완료 처리

In [ ]:
def node_transform_to_db(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return {**state, "db_rows": []}

    rows: List[DBRow] = []
    for o in extracted.orders:
        if not o.base_date:
            # base_date 없으면 적재 불가 → 스킵(issues에 이미 남김)
            continue

        base = datetime.strptime(o.base_date, "%Y-%m-%d").date()
        amt = parse_decimal_amount(o.transfer_amount)

        order_type = o.order_type
        if order_type not in ["SUB", "RED"]:
            order_type = infer_order_type_from_amount(amt)

        t_day = o.t_day
        if o.settle_class == "CONFIRMED":
            t_day = 0
        elif o.settle_class == "PENDING":
            if t_day is not None and t_day < 1:
                t_day = None

        rows.append(
            DBRow(
                task_id=state["task_id"],
                fund_code=o.fund_code.strip(),
                fund_name=o.fund_name.strip(),
                settle_class=o.settle_class,
                order_type=order_type,
                base_date=base,
                t_day=t_day,
                transfer_amount=abs_decimal(amt),  # 방향은 order_type, 금액은 절대값 저장(권장)
            )
        )

    return {**state, "db_rows": [r.model_dump() for r in rows]}

def route_after_validate(state: RState) -> str:
    if state["validation_ok"]:
        return "transform"
    if state["iterations"] >= 2:
        return "transform"
    return "repair"
    

### Graph 구성 및 빌드

In [ ]:
def build_file_graph():
    g = StateGraph(RState)
    g.add_node("ingest", node_ingest)
    g.add_node("extract", node_extract_value_only)
    g.add_node("validate", node_validate)
    g.add_node("repair", node_repair)
    g.add_node("transform", node_transform_to_db)

    g.add_edge(START, "ingest")
    g.add_edge("ingest", "extract")
    g.add_edge("extract", "validate")
    g.add_conditional_edges("validate", route_after_validate, {
        "repair": "repair",
        "transform": "transform",
    })
    g.add_edge("repair", "validate")
    g.add_edge("transform", END)
    return g.compile()
    

### Graph 호출

In [ ]:
# ============================================================
# 7) 외부 호출 API: 파일 1개 
# ============================================================
def extract_one_file(task_id: int, file_path: str, password: str) -> Dict[str, Any]:
    app = build_file_graph()

    init_state: RState = {
        "task_id": task_id,
        "file_path": file_path,
        "password": password,
        "markdown_doc": "",
        "ingest_issues": [],
        "extract_prompt": "",
        "extracted": None,
        "validation_ok": False,
        "validation_report": "",
        "iterations": 0,
        "db_rows": [],
        "extract_issues": [],
    }

    out = app.invoke(init_state)

    return {
        "file": Path(file_path).name,
        "db_rows": out.get("db_rows", []),
        "ingest_issues": out.get("ingest_issues", []),
        "extract_issues": out.get("extract_issues", []),
        "validation_report": out.get("validation_report", ""),
    }



### Graph 실행

In [ ]:
# ============================================================
# 8) 실행 예시
# ============================================================
# graph 실행

# text 추출 태스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
_password = '345678'
res = extract_one_file(1, _document_file_path, _password)


In [ ]:
visualize_res_result(res)

In [ ]:
res